# 01 — Huấn luyện detector theo cấu hình

Notebook này là giao diện mỏng cho workflow YOLO dùng chung. Preflight gọi cùng API với CLI `scripts/train_detector.py`; huấn luyện thật vẫn đi qua CLI để giữ nguyên kiểm tra version, policy dữ liệu và quy tắc không ghi đè. Không có vòng lặp huấn luyện hoặc cách tính metric nào được chép vào notebook.

Detector hiện có 9 lớp thương hiệu theo thứ tự cố định. Việc thêm lớp detector luôn đòi hỏi huấn luyện và validation một checkpoint tương thích; chỉ sửa registry không làm model nhận ra thương hiệu mới.

## Khởi chạy

```powershell
Set-Location D:/code/anhoa/train_embedding
./.venv/Scripts/Activate.ps1
jupyter lab notebooks/01_detector_training.ipynb
```

Hãy chạy `00_dataset_audit.ipynb` trước. Notebook không tự cài hoặc nâng cấp Ultralytics.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir() or not (PROJECT_ROOT / "configs").is_dir():
    raise RuntimeError("Hãy khởi chạy Jupyter từ thư mục gốc của repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display

from scripts.train_detector import preflight as cli_detector_preflight
from src.config import load_brand_registry, resolve_repo_path
from src.training.yolo import load_dataset_class_mapping, read_yolo_config

def show_table(rows: list[dict], title: str) -> None:
    display(Markdown(f"**{title}**"))
    if not rows:
        display(Markdown("_Không có dòng dữ liệu._"))
        return
    columns = list(rows[0])
    clean = lambda value: str(value).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    lines.extend("| " + " | ".join(clean(row.get(column, "")) for column in columns) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

## Cấu hình chạy

Mọi cờ thực thi mặc định là `False`. Validation là đánh giá mặc định; test có cờ độc lập và chỉ được mở sau khi lựa chọn checkpoint đã hoàn tất. Đổi `EVALUATION_OUTPUT_ROOT` hoặc xóa có chủ đích output thử nghiệm trước khi chạy lại vì notebook từ chối ghi đè.

In [ ]:
# Select a reviewed detector config; the same file is passed to the CLI.
# Legacy option: configs/brand_9brands_yolo.yaml
DETECTOR_CONFIG_PATH = "configs/brand_21brands_yolo.yaml"
BRAND_REGISTRY_PATH = "configs/brands.yaml"
ACTIVE_EVALUATION_WEIGHTS = "runs/model1_brand21/yolov8n_100ep_s42/weights/best.pt"
EVALUATION_OUTPUT_ROOT = "outputs/notebook_detector_evaluation"
EVALUATION_DEVICE = "0"
DEFAULT_EVALUATION_SPLIT = "val"

RUN_PREFLIGHT = False
RUN_TRAINING = False
RUN_VALIDATION_EVALUATION = False
RUN_TEST_EVALUATION = False

assert DEFAULT_EVALUATION_SPLIT == "val"
show_table(
    [
        {"tham số": "detector config", "giá trị": DETECTOR_CONFIG_PATH},
        {"tham số": "weights để đánh giá", "giá trị": ACTIVE_EVALUATION_WEIGHTS},
        {"tham số": "split mặc định", "giá trị": DEFAULT_EVALUATION_SPLIT},
        {"tham số": "device đánh giá", "giá trị": EVALUATION_DEVICE},
    ],
    "Lựa chọn hiện tại",
)

## Xem cấu hình hiệu lực và class order

Ô này chỉ đọc YAML. Class order từ dataset phải khớp registry 9 thương hiệu theo từng chỉ số. `evaluate_test_split: false`, `resume: false` và `exist_ok: false` là các guard an toàn của workflow hiện tại.

In [ ]:
config_path = resolve_repo_path(DETECTOR_CONFIG_PATH)
config = read_yolo_config(config_path)
data_yaml = resolve_repo_path(config["data"])
dataset_mapping = load_dataset_class_mapping(data_yaml)
registry = load_brand_registry(resolve_repo_path(BRAND_REGISTRY_PATH))
brand_profile = str(config.get("brand_profile", "brand_9brands"))
class_order = str(config.get("class_order", "registry"))
profile_brands = registry.profile(brand_profile)
registry_names = tuple(brand.name for brand in profile_brands)
registry_ids_by_name = {brand.name: brand.id for brand in profile_brands}
training_names = tuple(
    dataset_mapping[str(class_id)] for class_id in range(len(dataset_mapping))
)
if class_order == "source_data_yaml":
    assert set(training_names) == set(registry_names)
elif class_order == "registry":
    assert training_names == registry_names
else:
    raise ValueError(f"Unsupported class_order: {class_order!r}")

config_rows = [
    {"nhóm": "run", "tham số": "model", "giá trị": config["model"]},
    {"nhóm": "run", "tham số": "data", "giá trị": config["data"]},
    {"nhóm": "run", "tham số": "output", "giá trị": f"{config['project']}/{config['name']}"},
    {"nhóm": "run", "tham số": "brand_profile", "giá trị": brand_profile},
]
config_rows.extend({"nhóm": "train", "tham số": key, "giá trị": value} for key, value in config["train"].items())
config_rows.extend(
    [
        {"nhóm": "compatibility", "tham số": "verified_ultralytics", "giá trị": config["compatibility"]["verified_ultralytics"]},
        {"nhóm": "artifacts", "tham số": "evaluate_test_split", "giá trị": config["artifacts"]["evaluate_test_split"]},
    ]
)
show_table(config_rows, "Cấu hình detector")

mapping_rows = [
    {
        "training_id": class_id,
        "data.yaml": training_name,
        "stable_registry_id": registry_ids_by_name[training_name],
        "registry_name": training_name,
    }
    for class_id, training_name in enumerate(training_names)
]
show_table(mapping_rows, "Đối chiếu class order")
assert len({row["stable_registry_id"] for row in mapping_rows}) == len(training_names)

## Số lượng train/validation và provenance

Các bảng đọc `dataset_checks.json` đã tạo trước đó; không quét ảnh, không mở test và không sửa split. Quarantine vẫn là bằng chứng cần giữ, không phải dữ liệu để tự động nhập lại.

In [ ]:
checks_path = data_yaml.parent / "dataset_checks.json"
checks = json.loads(checks_path.read_text(encoding="utf-8"))
count_rows = [
    {
        "brand": brand,
        "train_images": checks["splits"]["train"]["per_brand"][brand]["images"],
        "train_instances": checks["splits"]["train"]["per_brand"][brand]["instances"],
        "train_families": checks["splits"]["train"]["per_brand"][brand]["families"],
        "val_images": checks["splits"]["val"]["per_brand"][brand]["images"],
        "val_instances": checks["splits"]["val"]["per_brand"][brand]["instances"],
        "val_families": checks["splits"]["val"]["per_brand"][brand]["families"],
    }
    for brand in training_names
]
show_table(count_rows, "Phân bố detector")
show_table(
    [
        {"kiểm tra": "cross_split_family_overlap", "giá trị": checks.get("cross_split_family_overlap")},
        {"kiểm tra": "exact_image_cross_split_overlap", "giá trị": checks.get("exact_image_cross_split_overlap")},
        {"kiểm tra": "known_holdout_violations", "giá trị": checks.get("known_holdout_violations")},
        {"kiểm tra": "quarantined_hashes_in_splits", "giá trị": checks.get("quarantined_hashes_in_splits")},
        {"kiểm tra": "unresolved_annotation_hashes", "giá trị": checks.get("unresolved_annotation_hashes")},
        {"kiểm tra": "test_split_evaluated", "giá trị": checks.get("test_split_evaluated")},
    ],
    "Tóm tắt integrity",
)
show_table(
    [
        {"artifact": path.relative_to(PROJECT_ROOT).as_posix(), "tồn tại": path.is_file(), "bytes": path.stat().st_size if path.is_file() else 0}
        for path in (checks_path, data_yaml.parent / "dataset_report.json", data_yaml.parent / "source_family_manifest.json", data_yaml.parent / "provenance.csv")
    ],
    "Artifact provenance",
)

In [ ]:
import matplotlib.pyplot as plt

brands = [row["brand"] for row in count_rows]
positions = list(range(len(brands)))
width = 0.42
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar([x - width / 2 for x in positions], [row["train_instances"] for row in count_rows], width, label="train")
ax.bar([x + width / 2 for x in positions], [row["val_instances"] for row in count_rows], width, label="val")
ax.set_xticks(positions, brands, rotation=35, ha="right")
ax.set_ylabel("Số bbox")
ax.set_title("Số bbox dùng cho huấn luyện và validation")
ax.legend()
fig.tight_layout()
plt.show()

## Preflight

Đây là cùng hàm `preflight` mà CLI sử dụng. Nó xác minh metadata, clean-subset policy, version được khai báo và đường dẫn output mới; nó không import backend Ultralytics và không khởi động huấn luyện.

In [ ]:
if RUN_PREFLIGHT:
    training_plan, preflight_summary = cli_detector_preflight(config_path)
    show_table([{"trường": key, "giá trị": value} for key, value in preflight_summary.items()], "Kết quả preflight")
else:
    training_plan = None
    print("Preflight đang tắt: RUN_PREFLIGHT=False")

## Huấn luyện thủ công

Đặt `RUN_PREFLIGHT=True` và kiểm tra bảng trước. Khi chủ động đặt `RUN_TRAINING=True`, ô này gọi CLI dùng chung. Tên run và output lấy từ YAML; CLI từ chối `resume`, `exist_ok` và run directory đã tồn tại.

In [ ]:
training_command = [
    sys.executable,
    str(resolve_repo_path("scripts/train_detector.py")),
    "--config",
    str(config_path),
]
print("Lệnh đã chuẩn bị:", subprocess.list2cmdline(training_command))
if RUN_TRAINING:
    if not RUN_PREFLIGHT:
        raise RuntimeError("Bật RUN_PREFLIGHT và xem kết quả trước khi huấn luyện.")
    subprocess.run(training_command, cwd=PROJECT_ROOT, check=True)
else:
    print("Không huấn luyện: RUN_TRAINING=False")

## Đánh giá validation hoặc test

Metric được Ultralytics tính trực tiếp; notebook chỉ chọn split và hiển thị per-brand precision, recall, mAP50, mAP50–95. `RUN_VALIDATION_EVALUATION` và `RUN_TEST_EVALUATION` loại trừ nhau. Validation là mặc định; test là opt-in riêng và không được dùng để chọn checkpoint.

In [ ]:
if RUN_VALIDATION_EVALUATION and RUN_TEST_EVALUATION:
    raise ValueError("Chỉ đánh giá một split trong mỗi lần chạy.")

if RUN_VALIDATION_EVALUATION or RUN_TEST_EVALUATION:
    evaluation_split = "test" if RUN_TEST_EVALUATION else DEFAULT_EVALUATION_SPLIT
    evaluation_weights = resolve_repo_path(ACTIVE_EVALUATION_WEIGHTS)
    evaluation_output = resolve_repo_path(EVALUATION_OUTPUT_ROOT) / f"{evaluation_weights.stem}_{evaluation_split}"
    if not evaluation_weights.is_file():
        raise FileNotFoundError(evaluation_weights)
    if evaluation_output.exists():
        raise FileExistsError(f"Chọn output mới; không ghi đè: {evaluation_output}")
    from ultralytics import YOLO, __version__ as ultralytics_version
    expected_version = str(config["compatibility"]["verified_ultralytics"])
    if str(ultralytics_version) != expected_version:
        raise RuntimeError(f"Cần Ultralytics {expected_version}, hiện có {ultralytics_version}")
    evaluation_model = YOLO(str(evaluation_weights))
    checkpoint_names = tuple(str(evaluation_model.names[index]) for index in range(len(evaluation_model.names)))
    if checkpoint_names != training_names:
        raise ValueError(f"Checkpoint class order không khớp data.yaml: {checkpoint_names}")
    metrics = evaluation_model.val(
        data=str(data_yaml),
        split=evaluation_split,
        imgsz=int(config["train"]["imgsz"]),
        batch=int(config["train"]["batch"]),
        device=EVALUATION_DEVICE,
        project=str(evaluation_output.parent),
        name=evaluation_output.name,
        exist_ok=False,
        plots=True,
    )
    metric_rows = []
    for class_id, brand in enumerate(training_names):
        precision, recall, map50, map50_95 = metrics.box.class_result(class_id)
        metric_rows.append({"class_id": class_id, "brand": brand, "precision": float(precision), "recall": float(recall), "mAP50": float(map50), "mAP50-95": float(map50_95)})
    show_table(metric_rows, f"Metric detector — {evaluation_split}")
else:
    print("Không đánh giá: cả hai cờ validation/test đều False.")

## Sau khi chạy

Dùng `03_metrics_review.ipynb` để so sánh best/last và metric theo thương hiệu. Chỉ cập nhật backend sang weights mới sau khi class order, validation và hợp đồng response đã được kiểm tra. Việc có detector mới không tự tạo gallery cho thương hiệu đó.